# 02 — Reconstruct portfolio returns

Pull historical returns for every tradeable ticker via `YahooFinanceProvider`,
then reconstruct each portfolio's daily return series using *current weights
× historical underlying-asset returns* (design D2).

**Caveat (forward-looking, not realised):** the reconstructed series is what
*today's* book *would have* returned over history — it's not a real track
record. Stashaway statements don't expose a NAV history we could reconcile
against, so this notebook is a sanity check that returns reconstruction works,
not a backtest.


In [ ]:
from datetime import date
from pathlib import Path
import warnings

import pandas as pd
import plotly.graph_objects as go

from hailmary.allocation.book_config import ROLES
from hailmary.allocation.portfolios import Role, from_parsed
from hailmary.allocation.returns import portfolio_returns
from hailmary.allocation.statements import parse_statement
from hailmary.data.providers import YahooFinanceProvider
from hailmary.viz.theme import apply_theme

STATEMENT_PATH = Path('../../data/statements/2026-04 StashAway Monthly Statement.pdf')
START = date(2022, 1, 1)
END = date.today()

## Parse + resolve

In [ ]:
parsed = parse_statement(STATEMENT_PATH, use_cache=False)
portfolios = [
    from_parsed(pf, roles=ROLES[pf.name])
    for pf in parsed if pf.name in ROLES
]
holding = [p for p in portfolios if Role.HOLDING in p.roles]
print(f'{len(portfolios)} portfolios resolved, {len(holding)} tagged HOLDING')

## Fetch returns for the diagnostic universe

Pull every Yahoo-resolvable ticker across HOLDING portfolios in one call so
the cache key is shared across portfolios.

In [ ]:
provider = YahooFinanceProvider()
tickers = sorted({
    h.metadata.ticker for p in holding for h in p.holdings
    if not h.metadata.ticker.startswith('CASH_')
})
print(f'Fetching {len(tickers)} unique tickers from Yahoo ({START} → {END})')
returns = provider.get_returns(tickers, START, END)
print(f'Returns panel: {returns.shape[0]} dates × {returns.shape[1]} tickers')
missing = sorted(set(tickers) - set(returns.columns))
if missing:
    print(f'\nTickers with NO Yahoo data: {missing}')
    print('These holdings will be excluded from return reconstruction; consider mapping to a different proxy in universe.py.')

## Reconstruct one return series per portfolio

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore', UserWarning)
    series = []
    failed = []
    for p in holding:
        try:
            s = portfolio_returns(p, returns=returns)
            series.append(s)
        except Exception as exc:
            failed.append((p.name, str(exc)))

panel = pd.concat(series, axis=1) if series else pd.DataFrame()
print(f'Reconstructed {len(series)} portfolio return series across {panel.shape[0]} dates')
if failed:
    print('\nFailed:')
    for n, e in failed:
        print(f'  {n}: {e}')

## Summary stats

In [ ]:
from hailmary.analytics.metrics import PerformanceMetrics
stats = []
common = panel.dropna(how='any')
for col in common.columns:
    m = PerformanceMetrics(common[col])
    stats.append({
        'portfolio': col,
        'ann_return': m.annualised_return,
        'ann_vol': m.annualised_vol,
        'sharpe': m.sharpe,
        'max_dd': m.max_drawdown,
    })
stats_df = pd.DataFrame(stats).set_index('portfolio').sort_values('sharpe', ascending=False)
stats_df.style.format({
    'ann_return': '{:.2%}', 'ann_vol': '{:.2%}', 'sharpe': '{:.2f}', 'max_dd': '{:.2%}'
})

## Cumulative-return chart

In [ ]:
cum = (1 + common).cumprod() - 1
fig = go.Figure()
for col in cum.columns:
    fig.add_trace(go.Scatter(x=cum.index, y=cum[col], mode='lines', name=col))
fig.update_layout(yaxis_tickformat='.0%')
apply_theme(fig, title='Reconstructed cumulative returns', height=520)

## Tracking-error placeholder

Stashaway statements don't expose a NAV history per portfolio, so we can't
compute a real tracking error against the parser's reconstruction. If a NAV
feed becomes available later, plot `(reconstructed - actual)` here.